# Module 25 — Procedural memory, and why a store that only grows degrades

**THE ONE IDEA:** two things almost nobody builds.

**Procedural memory** — the agent remembers *how* to do a task, not just facts about it.
A successful trajectory is distilled into a reusable skill.

**Forgetting** — decay and eviction. A store that only grows gets *worse*, because
retrieval precision falls as the ratio of noise to signal rises. **Forgetting is as
important as remembering**, and it is the part that gets skipped.

No API key — deterministic and free.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import math
from dataclasses import dataclass, field
from datetime import datetime, timedelta

@dataclass
class Skill:
    trigger: str
    steps: list                      # the distilled tool sequence
    successes: int = 0
    failures: int = 0
    last_used: datetime = field(default_factory=datetime.now)

    @property
    def reliability(self):           # Laplace-smoothed, so 1/1 is not 100%
        return (self.successes + 1) / (self.successes + self.failures + 2)

    def score(self, now, half_life_days=30):
        """Reliability decayed by recency. A skill that worked once, a year ago,
        should NOT outrank one that works every week."""
        age = (now - self.last_used).days
        return self.reliability * math.exp(-age * math.log(2) / half_life_days)

SKILLS: list[Skill] = []

## Distil a skill from a successful trajectory

Module 17 recorded traces. A trace that succeeded is exactly the raw material for a
skill: the tool sequence that worked.

In [ ]:
def distil(trigger, trace):
    steps = [s["tool"] for s in trace if s["kind"] == "tool"]
    sk = Skill(trigger=trigger, steps=steps, successes=1)
    SKILLS.append(sk); return sk

distil("compute an early repayment charge",
       [{"kind": "tool", "tool": "search_policy"}, {"kind": "tool", "tool": "calculate"}])
distil("check first-time-buyer eligibility",
       [{"kind": "tool", "tool": "search_policy"}, {"kind": "tool", "tool": "search_policy"}])

for s in SKILLS:
    print(f"  {s.trigger[:38]:38} -> {' -> '.join(s.steps)}")

## Use it as scaffolding, and record the outcome

In [ ]:
def recall(task, now=None):
    now = now or datetime.now()
    hits = [s for s in SKILLS if any(w in task.lower() for w in s.trigger.split()[:3])]
    return max(hits, key=lambda s: s.score(now), default=None)

sk = recall("compute an early repayment charge for MX-7741")
print("recalled:", sk.steps, f"reliability={sk.reliability:.2f}")

for outcome in [True, True, False, True]:          # feedback closes the loop
    sk.successes += outcome; sk.failures += (not outcome); sk.last_used = datetime.now()
print(f"after 4 more runs: {sk.successes}W/{sk.failures}L  reliability={sk.reliability:.2f}")

## Decay and eviction

In [ ]:
now = datetime.now()
SKILLS.append(Skill("handle a buy-to-let stress test", ["search_policy"],
                    successes=1, last_used=now - timedelta(days=400)))
SKILLS.append(Skill("apply a pandemic payment holiday", ["search_policy"],
                    successes=9, failures=0, last_used=now - timedelta(days=900)))

print(f"{'skill':40} {'rel':>5} {'age d':>6} {'score':>7}")
print("-" * 62)
for s in sorted(SKILLS, key=lambda s: -s.score(now)):
    print(f"{s.trigger[:40]:40} {s.reliability:5.2f} {(now - s.last_used).days:6} "
          f"{s.score(now):7.4f}")

EVICT = 0.05
dead = [s for s in SKILLS if s.score(now) < EVICT]
SKILLS[:] = [s for s in SKILLS if s.score(now) >= EVICT]
print(f"\nevicted {len(dead)}: {[s.trigger[:34] for s in dead]}")
print(f"{len(SKILLS)} skills retained")

print("""
LESSON - two tiers almost nobody builds.

PROCEDURAL memory closes the loop. The agent does not just know facts about the
user, it knows WHICH TOOL SEQUENCE WORKED for this shape of task, with a success
record attached. That is the substrate for module 28's Reflexion: fail, reflect,
store the correction, and do better next time WITHOUT retraining anything.

FORGETTING is the part that gets skipped, and skipping it degrades the system:

  - reliability is Laplace-smoothed, so one lucky success is not 100%. Without
    that, the newest skill always wins and nothing ever gets tested.
  - score decays with a HALF-LIFE. The pandemic payment-holiday skill has a
    perfect 9-0 record and is still correctly evicted, because the world moved on.
    A pure success-rate ranking would have kept it forever and confidently
    applied a policy that no longer exists.
  - eviction keeps the store small, so retrieval precision stays high. This is
    the same argument as module 23's 'write less than you think you need'.

The general rule for every memory tier in Block H: a store that only grows gets
WORSE, because precision falls as the noise-to-signal ratio rises. Decide the
forgetting policy when you build the store, not after it stops working.""")

---

**Next:** Block I — `../I_planning/26_plan_and_execute.ipynb`